# AI Assignment by Subhayan Das

## Import libraries

In [1]:
import pygame
import random
import time
import sys
from collections import deque
import numpy as np
import pandas as pd
import os
from PIL import Image, ImageDraw

pygame 2.6.1 (SDL 2.28.4, Python 3.9.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Constants

In [2]:
# Constants
GRID_SIZE = 30
CELL_SIZE = 5
WINDOW_SIZE = CELL_SIZE * (2 * GRID_SIZE + 1)
BACKGROUND_COLOR = (255, 255, 255)
WALL_COLOR = (0, 0, 0)
PATH_COLOR = (255, 255, 255)
START_COLOR = (0, 255, 0)
END_COLOR = (255, 0, 0)
PATHFIND_COLOR = (0, 0, 255)
VISITED_COLOR = (255, 255, 0)


## Path Finding Functions

In [3]:
def calculate_metrics(maze, start, end, screen, traversal_order, explored_nodes, path, start_time):
    def path_complexity(maze):
        dead_ends = 0
        branching_points = 0
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        
        for r in range(1, len(maze) - 1, 2):
            for c in range(1, len(maze[0]) - 1, 2):
                if maze[r][c] == 0:
                    open_neighbors = 0
                    for dr, dc in directions:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]) and maze[nr][nc] == 0:
                            open_neighbors += 1
                    if open_neighbors == 0:
                        dead_ends += 1
                    elif open_neighbors > 2:
                        branching_points += 1
        return dead_ends, branching_points

    dead_ends, branching_points = path_complexity(maze)

    total_cells = len(maze) * len(maze[0])
    walls = sum(row.count(1) for row in maze)
    open_cells = total_cells - walls
    wall_density = walls / total_cells

    execution_time = time.time() - start_time

    final_path_length = len(path)

    memory_usage = sys.getsizeof(maze) + sys.getsizeof(path) + sys.getsizeof(explored_nodes)

    return {
        "Algorithm" : "DFS",
        "Grid Size" : GRID_SIZE,
        "Path Complexity (Entropy)": (dead_ends, branching_points),
        "Wall Density (Open Space Ratio)": wall_density,
        "Number of Explored Nodes (Search Cost)": len(explored_nodes),
        "Final Path Length": final_path_length,
        "Execution Time (Seconds)": execution_time,
        "Memory Usage (Bytes)": memory_usage
    }

In [4]:
def dfs_traversal(maze, screen):
    start = (1, 1)
    end = (len(maze) - 2, len(maze[0]) - 2)
    
    stack = [start]
    visited = set()
    visited.add(start)
    parent = {}

    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]

    traversal_order = []
    while stack:
        r, c = stack.pop()
        traversal_order.append((r, c))

        if (r, c) == end:
            break  

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and maze[nr][nc] == 0:
                stack.append((nr, nc))
                visited.add((nr, nc))
                parent[(nr, nc)] = (r, c)

    path = []
    r, c = end
    while (r, c) != start:
        path.append((r, c))
        r, c = parent.get((r, c), start)  
    path.reverse()

    # Draw traversal
    for cell in traversal_order:
        draw_maze(screen, maze, [], visited_cells=traversal_order[:traversal_order.index(cell) + 1])
        pygame.display.flip()
        pygame.time.delay(1)  

    # Draw the final path in blue
    draw_maze(screen, maze, path)
    pygame.display.flip()

    return path, traversal_order, visited

In [5]:
def draw_maze(screen, maze, path, visited_cells=[]):
    for r in range(len(maze)):
        for c in range(len(maze[0])):
            color = WALL_COLOR if maze[r][c] == 1 else PATH_COLOR
            pygame.draw.rect(screen, color, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in visited_cells:
        pygame.draw.rect(screen, VISITED_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in path:
        pygame.draw.rect(screen, PATHFIND_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    pygame.draw.rect(screen, START_COLOR, (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE))
    pygame.draw.rect(screen, END_COLOR, ((len(maze[0]) - 2) * CELL_SIZE, (len(maze) - 2) * CELL_SIZE, CELL_SIZE, CELL_SIZE))


In [6]:
def export_path(maze, path, start, end, filename):
    rows, cols = len(maze), len(maze[0])
    image = Image.new("RGB", (cols * CELL_SIZE, rows * CELL_SIZE), (255, 255, 255))
    draw = ImageDraw.Draw(image)

    for r in range(rows):
        for c in range(cols):
            if maze[r][c] == 1:  # Wall
                draw.rectangle([c * CELL_SIZE, r * CELL_SIZE, (c + 1) * CELL_SIZE, (r + 1) * CELL_SIZE], fill=(0, 0, 0))

    for r, c in path:
        draw.rectangle([c * CELL_SIZE, r * CELL_SIZE, (c + 1) * CELL_SIZE, (r + 1) * CELL_SIZE], fill=(0, 0, 255))

    start_r, start_c = start
    draw.rectangle([start_c * CELL_SIZE, start_r * CELL_SIZE, (start_c + 1) * CELL_SIZE, (start_r + 1) * CELL_SIZE], fill=(0, 255, 0))

    end_r, end_c = end
    draw.rectangle([end_c * CELL_SIZE, end_r * CELL_SIZE, (end_c + 1) * CELL_SIZE, (end_r + 1) * CELL_SIZE], fill=(255, 0, 0))

    image.save(filename)
    print(f"Final path image saved as {filename}")

## Main Function

In [7]:
def main():
    pygame.init()

    screen_dfs = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE))
    pygame.display.set_caption("DFS Maze Traversal")

    maze = np.load(f"./Mazes/{GRID_SIZE}.npy").tolist()

    start_time_dfs = time.time()
    dfs_path, dfs_traversal_order, dfs_visited = dfs_traversal(maze, screen_dfs)
    dfs_metrics = calculate_metrics(maze, (1, 1), (len(maze) - 2, len(maze[0]) - 2), screen_dfs, dfs_traversal_order, dfs_visited, dfs_path, start_time_dfs)

    
    print("DFS Metrics:")
    for metric, value in dfs_metrics.items():
        print(f"{metric}: {value}")

    # Saving the metrics:
    filename = 'results.csv'
    df = pd.DataFrame([dfs_metrics])

    if os.path.exists(filename):
        existing_df = pd.read_csv(filename)
        df = pd.concat([existing_df, df], ignore_index=True)
    
    df.to_csv(filename, index=False)

    export_path(maze, dfs_path, (1, 1), (len(maze) - 2, len(maze[0]) - 2), f"./Paths/DFS_{GRID_SIZE}.png")
    
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        pygame.display.flip()

    pygame.quit()

In [8]:

if __name__ == "__main__":
    main()

DFS Metrics:
Algorithm: DFS
Grid Size: 30
Path Complexity (Entropy): (0, 485)
Wall Density (Open Space Ratio): 0.44396667562483205
Number of Explored Nodes (Search Cost): 1843
Final Path Length: 908
Execution Time (Seconds): 3.709855794906616
Memory Usage (Bytes): 139664
Final path image saved as ./Paths/DFS_30.png
